## OilyGiant Company

The company operates in oil extraction, and the main task is to find the best locations to **open 200 new oil wells while minimizing risks and maximizing profits**.



### 1. Data Loading and Preparation



The three datasets were loaded (`geo_data_0.csv`, `geo_data_1.csv`, `geo_data_2.csv`), each containing information about oil wells.
- Each dataset contains 100,000 records and 5 columns:
  - id: unique well identifier (not used in modeling).
  - f0, f1, f2: three numerical characteristics of the wells. According to the initial project information, their specific meaning is not important, but the features themselves are significant.
  - **product: oil reserve volume in the well (in thousand barrels), which is the target variable.**


- Data types were verified: f0, f1, f2, and product are float64, while id is object.
- No missing values or duplicates were found in any region.

Descriptive analysis:
- Region 0:
  - Average reserves: 92.5 thousand barrels.
  - Relatively symmetric distribution, ranging from 0 to 185 thousand.

- Region 1:
  - Average reserves: 68.8 thousand barrels.
  - High dispersion in the features (f0 and f1 have very large positive and negative values).
  - Reserves range from 0 to 137 thousand.

- Region 2:
  - Average reserves: 95 thousand barrels.
  - Stable distribution, with reserves between 0 and 190 thousand.


- The data is clean and does not require initial imputation or cleaning.
- The id variable is discarded for modeling.
- f0, f1, and f2 are kept as features.
- product is the target variable to predict.
- At first glance, Region 2 appears to have a slightly higher average reserve volume than the others, but this must be confirmed with the model and profit analysis.



In [27]:
# Import the required libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error



In [28]:
# Load the three datasets
data0 = pd.read_csv('datasets/geo_data_0.csv')
data1 = pd.read_csv('datasets/geo_data_1.csv')
data2 = pd.read_csv('datasets/geo_data_2.csv')

# Basic inspection
for i, d in enumerate([data0, data1, data2]):
    print(f"--- Region {i} ---")
    print(d.info())
    print(d.head())
    print(d.describe())
    print("Missing values:", d.isna().sum().sum())
    print("Duplicates:", d.duplicated().sum())
    print()



--- Region 0 ---
<class 'pandas.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 5 columns):
 #   Column   Non-Null Count   Dtype  
---  ------   --------------   -----  
 0   id       100000 non-null  str    
 1   f0       100000 non-null  float64
 2   f1       100000 non-null  float64
 3   f2       100000 non-null  float64
 4   product  100000 non-null  float64
dtypes: float64(4), str(1)
memory usage: 3.8 MB
None
      id        f0        f1        f2     product
0  txEyH  0.705745 -0.497823  1.221170  105.280062
1  2acmU  1.334711 -0.340164  4.365080   73.037750
2  409Wp  1.022732  0.151990  1.419926   85.265647
3  iJLyR -0.032172  0.139033  2.978566  168.620776
4  Xdl7t  1.988431  0.155413  4.751769  154.036647
                  f0             f1             f2        product
count  100000.000000  100000.000000  100000.000000  100000.000000
mean        0.500419       0.250143       2.502647      92.500000
std         0.871832       0.504433       3.248248     

### 2. Train and Test the Model for Each Region in *geo_data_0.csv*



In [29]:
# separate features and target
features0 = data0.drop(['id', 'product'], axis=1)
target0 = data0['product']

# 2.1 split into train and validation sets (75:25)
features_train, features_valid, target_train, target_valid = train_test_split(
    features0, target0, test_size=0.25, random_state=12345
)

# 2.2 train model
model = LinearRegression()
model.fit(features_train, target_train)

# 2.3 predictions
predictions = model.predict(features_valid)

# 2.4 calculate metrics
rmse = root_mean_squared_error(target_valid, predictions)
mean_pred = predictions.mean()

print("Average predicted reserve volume:", mean_pred)
print("Model RMSE:", rmse)



Average predicted reserve volume: 92.59256778438038
Model RMSE: 37.5794217150813


**Results for Region 0 (geo_data_0.csv)**
- Average predicted reserve volume: 92.6 thousand barrels.
    This is very aligned with the actual regional average (92.5 thousand barrels). It indicates that the model does not have systematic bias: it performs fairly well on average.
  
- Root Mean Squared Error (RMSE): 37.6 thousand barrels.
    The error is significant. Compared with the average (92.5), the model can be off by more than one third of the expected value. This confirms that linear regression does not predict each individual well with high precision.



In [30]:
# 2.6 repeat and execute steps 2.1-2.5 for the files
# 'geo_data_1.csv' and 'geo_data_2.csv'.

def train_and_validate(path, region_name):
    data = pd.read_csv(path)
    features = data.drop(['id', 'product'], axis=1)
    target = data['product']
    
    # split
    X_train, X_valid, y_train, y_valid = train_test_split(
        features, target, test_size=0.25, random_state=12345
    )
    
    # train
    model = LinearRegression()
    model.fit(X_train, y_train)
    
    # predictions
    predictions = model.predict(X_valid)
    
    # metrics
    rmse = root_mean_squared_error(y_valid, predictions)
    mean_pred = predictions.mean()
    
    print(f"--- Results {region_name} ---")
    print("Average predicted reserve volume:", mean_pred)
    print("RMSE:", rmse)
    print()
    
    return predictions, y_valid, model

# execute for the three regions
pred0, y0, model0 = train_and_validate('datasets/geo_data_0.csv', "Region 0")
pred1, y1, model1 = train_and_validate('datasets/geo_data_1.csv', "Region 1")
pred2, y2, model2 = train_and_validate('datasets/geo_data_2.csv', "Region 2")



--- Results Region 0 ---
Average predicted reserve volume: 92.59256778438038
RMSE: 37.5794217150813

--- Results Region 1 ---
Average predicted reserve volume: 68.72854689544603
RMSE: 0.8930992867756166

--- Results Region 2 ---
Average predicted reserve volume: 94.96504596800492
RMSE: 40.02970873393434



Region 0: The model predicts an average very close to the actual value (92.5), but the error is high (37.6). This means that individual predictions vary considerably, even though the average is well adjusted.

Region 1: The extremely low RMSE (0.89) stands out. This indicates that the model predicts almost perfectly on the validation set. However, this should be interpreted carefully: the data is generated in such a way that, in this region, the target variable depends almost exactly on one of the features, which is why linear regression fits with almost no error.
The issue is that the average reserve volume (68.7) is lower than in the other regions.

Region 2: The predicted average is very high (95.0), even higher than Region 0. The error (40.0) is also considerable, similar to Region 0, which shows dispersion in the predictions.



### 3. Profit Calculation

#### 3.1 Required Variables
According to the conditions:
- Total budget: 100 million USD
- Number of wells to develop: 200
- Price per 1k barrels: 4.5 USD per barrel



In [31]:
BUDGET = 100_000_000      # total budget in dollars
OIL_PRICE = 4.5           # USD per barrel
UNITS_PRICE = 4500        # USD per unit (thousand barrels)
NUMBER_OF_WELLS = 200     # wells to develop



#### 3.2 Profitability Threshold

The budget is distributed across 200 wells. Each well must generate 500,000 USD in revenue to avoid losses.

In terms of reserves: 500,000 / 4500 = 111.1. In other words, each well must have at least 111.1 thousand barrels of reserves.

Comparison with the previous results:
- Region 0: average = 92.6 -> below the threshold.
- Region 1: average = 68.7 -> far below the threshold.
- Region 2: average = 95.0 -> also below the threshold.

No region reaches the profitability threshold on average. Even so, the project is not decided based on the average: the 200 wells with the highest predicted reserves will be selected, not an average well.
This means that, although the average is below 111.1, selecting the best 200 wells may still exceed the threshold and generate profit.



### 4. Function to Calculate Profit and Select the Top 200 Wells by Region

4.1 and 4.2
- Validation predictions are sorted in descending order.
- The first 200 wells are selected.
- Using those indices, the actual reserve volume (`target_valid`) is recovered.
- Profit is calculated as follows:

  *Profit = Sum of actual reserves (Top 200) x 4500 - 100,000,000*



In [32]:
def calculate_profit(predictions, target, count=200, price=4500, budget=100_000_000):
    # convert to DataFrame to combine predictions with actual values
    df = pd.DataFrame({'predictions': predictions, 'target': target})
    
    # sort by predictions in descending order
    top_wells = df.sort_values(by='predictions', ascending=False).head(count)
    
    # actual reserve volume of the selected wells
    total_product = top_wells['target'].sum()
    
    # revenue
    revenue = total_product * price
    
    # net profit
    profit = revenue - budget
    
    return profit, top_wells['target'].mean(), total_product



#### 4.3 Apply to the Three Regions



In [33]:
profit0, mean0, total0 = calculate_profit(pred0, y0)
profit1, mean1, total1 = calculate_profit(pred1, y1)
profit2, mean2, total2 = calculate_profit(pred2, y2)

print("--- Region 0 ---")
print("Potential profit:", profit0)
print("Total volume (top 200):", total0)
print("Average volume (top 200):", mean0)
print()

print("--- Region 1 ---")
print("Potential profit:", profit1)
print("Total volume (top 200):", total1)
print("Average volume (top 200):", mean1)
print()

print("--- Region 2 ---")
print("Potential profit:", profit2)
print("Total volume (top 200):", total2)
print("Average volume (top 200):", mean2)



--- Region 0 ---
Potential profit: 33208260.43139851
Total volume (top 200): 29601.83565142189
Average volume (top 200): 148.00917825710945

--- Region 1 ---
Potential profit: 24150866.966815114
Total volume (top 200): 27589.081548181137
Average volume (top 200): 137.9454077409057

--- Region 2 ---
Potential profit: 27103499.635998324
Total volume (top 200): 28245.22214133296
Average volume (top 200): 141.2261107066648


#### Region 0 appears to be the most promising option because its top 200 wells project the highest potential profit and a higher average reserve volume than the other regions.

Region 1, despite having a very accurate model, does not reach the desired profitability threshold due to its low average reserve volume.

Region 2 shows intermediate behavior, but it does not outperform Region 0 in terms of potential profit.



### 5. Risk and Profit Analysis with Bootstrapping



In [34]:
# 5.1 Business and sampling parameters
BUDGET = 100_000_000            # USD
PRICE_PER_UNIT = 4500           # 4.5 USD/barrel * 1000 barrels
SAMPLE_SIZE = 500               # points explored per region
TOP_K = 200                     # wells to develop
N_BOOT = 1000                   # bootstrap repetitions
state = np.random.RandomState(12345)

def _profit_from_sample(preds, y_true, idx):
    """
    Given a sample of indices (idx):
      1) sort by prediction descending
      2) take TOP_K
      3) sum the actual product
      4) calculate profit = product_sum*PRICE_PER_UNIT - BUDGET
    """
    preds = np.asarray(preds)
    y_true = np.asarray(y_true)

    order = np.argsort(preds[idx])[::-1][:TOP_K]
    chosen = idx[order]
    total_product = y_true[chosen].sum()
    return total_product * PRICE_PER_UNIT - BUDGET

def bootstrap_profits(preds, y_true, n_boot=N_BOOT, sample_size=SAMPLE_SIZE, rng=state):
    """Return a Series with N_BOOT simulated profits for a region."""
    n = len(preds)
    profits = np.empty(n_boot, dtype=float)
    for b in range(n_boot):
        idx = rng.choice(n, size=sample_size, replace=True)
        profits[b] = _profit_from_sample(preds, y_true, idx)
    return pd.Series(profits, name="profit")

# Generate the profit distribution by region
profits0 = bootstrap_profits(pred0, y0)
profits1 = bootstrap_profits(pred1, y1)
profits2 = bootstrap_profits(pred2, y2)

# Quick check (to verify that they exist and contain 1000 samples)
print(profits0.head(), "
")
print(profits1.head(), "
")
print(profits2.head())



0    6.054641e+06
1    5.363934e+06
2    2.937858e+06
3    1.789934e+06
4    2.719929e+06
Name: profit, dtype: float64 

0    3.178949e+06
1    1.010041e+06
2    1.295538e+06
3    5.105900e+06
4    5.512065e+06
Name: profit, dtype: float64 

0    7.433808e+06
1    5.687995e+06
2    2.468884e+06
3    3.901355e+06
4    5.309689e+06
Name: profit, dtype: float64


Each list (`profits0`, `profits1`, `profits2`) contains 1000 simulated profit values for each region.

In each simulation, the following steps were performed:
- A sample of 500 wells was taken with replacement.
- The 200 wells with the highest predicted reserves were selected.
- Revenue was calculated using the actual value (`product`) of those 200 wells.
- The fixed budget of 100 million USD was subtracted.

The results show that:
- For Region 0, the first simulations show profits such as 6.05M, 5.36M, 2.94M... (in dollars).
- For Region 1, the values are lower, for example 3.17M, 1.01M, 1.29M...
- For Region 2, some simulations show higher profits, such as 7.43M, 5.68M, 2.47M...



In [35]:
# 5.2 Average profit

def summarize_bootstrap(profits, region_name):
    mean_profit = profits.mean()
    ci_low, ci_high = profits.quantile([0.025, 0.975])
    risk = (profits < 0).mean() * 100
    
    print(f"--- {region_name} ---")
    print(f"Average profit: {mean_profit:,.0f} USD")
    print(f"95% CI: [{ci_low:,.0f} ; {ci_high:,.0f}] USD")
    print(f"Risk of loss: {risk:.2f}%")
    print()
    
    return mean_profit, ci_low, ci_high, risk

res0 = summarize_bootstrap(profits0, "Region 0")
res1 = summarize_bootstrap(profits1, "Region 1")
res2 = summarize_bootstrap(profits2, "Region 2")



--- Region 0 ---
Average profit: 3,961,650 USD
95% CI: [-1,112,155 ; 9,097,669] USD
Risk of loss: 6.90%

--- Region 1 ---
Average profit: 4,611,558 USD
95% CI: [780,508 ; 8,629,521] USD
Risk of loss: 0.70%

--- Region 2 ---
Average profit: 3,929,505 USD
95% CI: [-1,122,276 ; 9,345,629] USD
Risk of loss: 6.50%



#### 5.3 According to the business criteria (high average profit and risk of loss < 2.5%), the only region that meets the requirements is Region 1.

This matches the fact that its model has the lowest error (RMSE ? 0.89) and predicts with high precision.

Although in step 4 Region 0 seemed more attractive because of its higher point-estimate profit, the risk evaluation shows that its volatility is too high.

Therefore, the final recommendation is to develop the wells in Region 1, because it combines safety (very low risk) with a higher average profit.



### Final Project Conclusion

After applying the linear regression model and evaluating the results with bootstrapping, the following conclusions can be drawn:

- Region 0: Although it showed attractive potential profit in the initial analysis, bootstrapping revealed a 6.9% risk of loss, far above the maximum accepted threshold of 2.5%. This makes the region unsuitable for investment because profit volatility is too high.

- Region 1: This region stands out as the most consistent. Average profit was 4.61 million USD, the 95% confidence interval remained positive ([0.78M ; 8.62M] USD), and the risk of loss was only 0.7%, well below the allowed limit. This confirms that, although the average reserve volume in this region is lower than in others, the high precision of the model ensures efficient selection of the most profitable wells.

- Region 2: Like Region 0, this region showed a high risk of loss (6.5%) and therefore does not meet the business conditions for development, despite having an average profit similar to Region 0.



## Dashboard Export

This final section exports the model and business results into a small JSON file used by the interactive portfolio dashboard. Run it after all previous analysis cells.


In [ ]:
# Dashboard Export
# Run this cell after the full notebook analysis to refresh dashboard/results.json.
import json
from pathlib import Path

regions = [
    {
        "id": "region-0",
        "name": "Region 0",
        "label": "Region 0",
        "predictions": pred0,
        "actual": y0,
        "profit_distribution": profits0,
        "recommendation": False,
    },
    {
        "id": "region-1",
        "name": "Region 1",
        "label": "Region 1",
        "predictions": pred1,
        "actual": y1,
        "profit_distribution": profits1,
        "recommendation": True,
    },
    {
        "id": "region-2",
        "name": "Region 2",
        "label": "Region 2",
        "predictions": pred2,
        "actual": y2,
        "profit_distribution": profits2,
        "recommendation": False,
    },
]

summary = []
prediction_samples = {}
for region in regions:
    predictions_array = np.asarray(region["predictions"])
    actual_array = np.asarray(region["actual"])
    profit_distribution = region["profit_distribution"]
    potential_profit, top_mean_volume, top_total_volume = calculate_profit(
        predictions_array,
        actual_array,
        count=TOP_K,
        price=PRICE_PER_UNIT,
        budget=BUDGET,
    )
    ci_low, ci_high = profit_distribution.quantile([0.025, 0.975])
    sample_size = min(650, len(predictions_array))
    sample_rng = np.random.RandomState(12345)
    sample_idx = sample_rng.choice(len(predictions_array), size=sample_size, replace=False)
    prediction_samples[region["id"]] = [
        {
            "actual": round(float(actual_array[idx]), 2),
            "predicted": round(float(predictions_array[idx]), 2),
        }
        for idx in sample_idx
    ]
    summary.append({
        "id": region["id"],
        "name": region["name"],
        "label": region["label"],
        "recommended": region["recommendation"],
        "mean_predicted_reserves": round(float(predictions_array.mean()), 2),
        "rmse": round(float(root_mean_squared_error(actual_array, predictions_array)), 2),
        "potential_profit_usd": round(float(potential_profit), 2),
        "top_200_total_reserves": round(float(top_total_volume), 2),
        "top_200_mean_reserves": round(float(top_mean_volume), 2),
        "bootstrap_mean_profit_usd": round(float(profit_distribution.mean()), 2),
        "confidence_interval_95_usd": [round(float(ci_low), 2), round(float(ci_high), 2)],
        "loss_risk_pct": round(float((profit_distribution < 0).mean() * 100), 2),
    })

recommended_region = next(region for region in summary if region["recommended"])
results = {
    "project": {
        "title": "Oil Well Investment Risk Analysis",
        "subtitle": "Machine learning and bootstrap simulation to select the best region for 200 new oil wells.",
        "objective": "Maximize expected profit while keeping loss risk below 2.5%.",
        "recommended_region": recommended_region["name"],
        "model": "Linear Regression",
        "validation_split": "75% train / 25% validation",
        "bootstrap_iterations": int(N_BOOT),
        "sample_size": int(SAMPLE_SIZE),
        "selected_wells": int(TOP_K),
        "budget_usd": int(BUDGET),
        "price_per_unit_usd": int(PRICE_PER_UNIT),
    },
    "regions": summary,
    "prediction_samples": prediction_samples,
}

output_path = Path("dashboard") / "results.json"
output_path.parent.mkdir(parents=True, exist_ok=True)
output_path.write_text(json.dumps(results, indent=2), encoding="utf-8")
print(f"Dashboard data exported to {output_path}")


